In [132]:
import pickle
import logging
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV, cross_validate
from sklearn.metrics import f1_score, accuracy_score, classification_report, \
confusion_matrix, precision_score, recall_score, roc_auc_score
from lightgbm import LGBMClassifier 
from sklearn.preprocessing import RobustScaler, MinMaxScaler, StandardScaler
from sklearn.utils import resample
from imblearn.over_sampling import SMOTE
import matplotlib.pyplot as plt
import itertools
%matplotlib inline

## Read soundtrack training data

In [133]:
soundtrack_df = pd.read_csv("../data/cleaned/final_movie_soundtrack_data_v2.csv",",").loc[:,'track_name':'movie_genre']
soundtrack_df.head()

,track_name,album_name,artist_names,release_date,duration_ms,popularity,song_genres,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,movie_genre
0,Space Exploration,Venom,Ludwig Göransson,10/5/18,263973,21,"nordic soundtrack,orchestral soundtrack,soundt...",0.194,0.294,8,-19.131,1,0.0513,0.605,0.890,0.0862,0.0333,159.468,4,"Action,Sci-Fi"
1,Symbiotes Arrive,Venom,Ludwig Göransson,10/5/18,123040,17,"nordic soundtrack,orchestral soundtrack,soundt...",0.440,0.177,0,-21.577,0,0.0340,0.897,0.925,0.1080,0.0663,124.030,4,"Action,Sci-Fi"
2,First Contact,Venom,Ludwig Göransson,10/5/18,209586,15,"nordic soundtrack,orchestral soundtrack,soundt...",0.194,0.153,0,-20.763,0,0.0471,0.820,0.962,0.1320,0.0338,117.213,4,"Action,Sci-Fi"
3,Eddie's Blues,Venom,Ludwig Göransson,10/5/18,290440,20,"nordic soundtrack,orchestral soundtrack,soundt...",0.224,0.205,1,-17.388,0,0.0490,0.566,0.937,0.1070,0.0372,119.046,4,"Action,Sci-Fi"
4,"Run, Eddie, Run",Venom,Ludwig Göransson,10/5/18,107546,20,"nordic soundtrack,orchestral soundtrack,soundt...",0.154,0.670,10,-12.670,0,0.0509,0.051,0.664,0.1760,0.0381,84.224,5,"Action,Sci-Fi"


In [134]:
soundtrack_df['movie_genre'].value_counts()

Action,Sci-Fi           510
Drama                   295
Action,Comedy           190
Comedy                  169
Action,Comedy,Sci-Fi    165
Comedy,Drama            146
Action,Drama            137
Horror                  135
Action                  124
Drama,Romance            90
Drama,Horror             78
Romance                  73
Drama,Sci-Fi             52
Comedy,Drama,Romance     42
Action,Drama,Sci-Fi      35
Comedy,Horror,Sci-Fi     33
Comedy,Romance           33
Comedy,Drama,Horror      24
Action,Romance           23
Action,Drama,Romance     22
Action,Horror,Sci-Fi     18
Name: movie_genre, dtype: int64

In [135]:
soundtrack_df[soundtrack_df['movie_genre']=='Comedy,Drama,Romance'][['album_name']].value_counts()

album_name       
Crazy Rich Asians    14
Focus                14
How To Be Single     14
dtype: int64

In [136]:
soundtrack_df.loc[soundtrack_df['album_name']=='IT Chapter Two','movie_genre'] ='Horror'

In [137]:
genre_replacements = {'Comedy,Drama':'Comedy'
                ,'Drama,Romance':'Romance'
                ,'Comedy,Romance':'Romance'
                ,'Drama,Horror':'Horror'
                ,'Action,Horror,Sci-Fi':'Horror'
                ,'Action,Sci-Fi':'Sci-Fi'
                ,'Action,Drama,Sci-Fi':'Action'
                ,'Action,Comedy,Sci-Fi':'Action'
                ,'Action,Drama':'Action'
                ,'Action,Comedy':'Action'
                ,'Comedy,Drama,Romance':'Comedy'
                ,'Action,Drama,Romance':'Drama'
                ,'Drama,Sci-Fi':'Sci-Fi'
               }

soundtrack_df['movie_genre'].replace(genre_replacements,inplace=True)
soundtrack_df['movie_genre'].value_counts()

Action                  606
Sci-Fi                  562
Comedy                  357
Drama                   317
Horror                  276
Romance                 196
Comedy,Horror,Sci-Fi     33
Comedy,Drama,Horror      24
Action,Romance           23
Name: movie_genre, dtype: int64

## Filter on single movie genre categories

In [138]:
single_genre_df = soundtrack_df[soundtrack_df['movie_genre'].isin(['Horror','Romance','Action','Comedy','Drama','Sci-Fi'])]
single_genre_df

,track_name,album_name,artist_names,release_date,duration_ms,popularity,song_genres,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,movie_genre
0,Space Exploration,Venom,Ludwig Göransson,10/5/18,263973,21,"nordic soundtrack,orchestral soundtrack,soundt...",0.194,0.2940,8,-19.131,1,0.0513,0.605,0.890,0.0862,0.0333,159.468,4,Sci-Fi
1,Symbiotes Arrive,Venom,Ludwig Göransson,10/5/18,123040,17,"nordic soundtrack,orchestral soundtrack,soundt...",0.440,0.1770,0,-21.577,0,0.0340,0.897,0.925,0.1080,0.0663,124.030,4,Sci-Fi
2,First Contact,Venom,Ludwig Göransson,10/5/18,209586,15,"nordic soundtrack,orchestral soundtrack,soundt...",0.194,0.1530,0,-20.763,0,0.0471,0.820,0.962,0.1320,0.0338,117.213,4,Sci-Fi
3,Eddie's Blues,Venom,Ludwig Göransson,10/5/18,290440,20,"nordic soundtrack,orchestral soundtrack,soundt...",0.224,0.2050,1,-17.388,0,0.0490,0.566,0.937,0.1070,0.0372,119.046,4,Sci-Fi
4,"Run, Eddie, Run",Venom,Ludwig Göransson,10/5/18,107546,20,"nordic soundtrack,orchestral soundtrack,soundt...",0.154,0.6700,10,-12.670,0,0.0509,0.051,0.664,0.1760,0.0381,84.224,5,Sci-Fi
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2389,Ritual casting,The Conjuring,joseph bishara,7/16/13,205880,24,soundtrack,0.169,0.1380,8,-25.104,1,0.0436,0.987,0.912,0.0723,0.0265,60.143,4,Horror
2390,cellar tone,The Conjuring,joseph bishara,7/16/13,58786,20,soundtrack,0.164,0.0236,2,-38.083,1,0.0438,0.901,0.868,0.1110,0.1170,128.004,4,Horror
2391,Annabelle,The Conjuring,joseph bishara,7/16/13,206346,19,soundtrack,0.138,0.1490,8,-21.586,1,0.0443,0.213,0.750,0.0857,0.0317,88.096,4,Horror
2392,doll box,The Conjuring,joseph bishara,7/16/13,46600,25,soundtrack,0.409,0.0652,8,-22.133,1,0.0939,0.895,0.784,0.2360,0.8540,207.542,4,Horror


In [139]:
# #Get aggregated values
single_genre_df.groupby("movie_genre").agg({'duration_ms': ['mean'],
                                              'danceability': ['mean'],
                                              'energy': ['mean'],
                                              'loudness': ['mean'],
                                              'acousticness': ['mean'],
                                              'instrumentalness': ['mean'],
                                              'valence': ['mean'],
                                              'tempo': ['mean']})

,duration_ms,danceability,energy,loudness,acousticness,instrumentalness,valence,tempo
,mean,mean,mean,mean,mean,mean,mean,mean
movie_genre,,,,,,,,
Action,186836.688119,0.319681,0.344945,-17.185678,0.490249,0.746817,0.150800,110.380906
Comedy,174320.081232,0.494255,0.419548,-13.258437,0.477991,0.428024,0.367918,114.521101
Drama,158602.012618,0.330050,0.262501,-19.370183,0.695405,0.704359,0.160423,105.593625
Horror,143571.851449,0.253522,0.222626,-20.174185,0.609093,0.706316,0.098266,102.892040
Romance,139477.030612,0.348605,0.254341,-18.862857,0.731267,0.534590,0.224839,106.229403
Sci-Fi,207137.320285,0.237300,0.254173,-18.919836,0.608059,0.806966,0.079508,104.069874


In [140]:
# Get genres into their own columns as binary variables
genre_dummies = pd.get_dummies(single_genre_df['movie_genre'])

# Concatenate the original DataFrame with the binary columns
combined_df = pd.concat([single_genre_df, genre_dummies], axis=1)
combined_df

,track_name,album_name,artist_names,release_date,duration_ms,popularity,song_genres,danceability,energy,key,...,valence,tempo,time_signature,movie_genre,Action,Comedy,Drama,Horror,Romance,Sci-Fi
0,Space Exploration,Venom,Ludwig Göransson,10/5/18,263973,21,"nordic soundtrack,orchestral soundtrack,soundt...",0.194,0.2940,8,...,0.0333,159.468,4,Sci-Fi,0,0,0,0,0,1
1,Symbiotes Arrive,Venom,Ludwig Göransson,10/5/18,123040,17,"nordic soundtrack,orchestral soundtrack,soundt...",0.440,0.1770,0,...,0.0663,124.030,4,Sci-Fi,0,0,0,0,0,1
2,First Contact,Venom,Ludwig Göransson,10/5/18,209586,15,"nordic soundtrack,orchestral soundtrack,soundt...",0.194,0.1530,0,...,0.0338,117.213,4,Sci-Fi,0,0,0,0,0,1
3,Eddie's Blues,Venom,Ludwig Göransson,10/5/18,290440,20,"nordic soundtrack,orchestral soundtrack,soundt...",0.224,0.2050,1,...,0.0372,119.046,4,Sci-Fi,0,0,0,0,0,1
4,"Run, Eddie, Run",Venom,Ludwig Göransson,10/5/18,107546,20,"nordic soundtrack,orchestral soundtrack,soundt...",0.154,0.6700,10,...,0.0381,84.224,5,Sci-Fi,0,0,0,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2389,Ritual casting,The Conjuring,joseph bishara,7/16/13,205880,24,soundtrack,0.169,0.1380,8,...,0.0265,60.143,4,Horror,0,0,0,1,0,0
2390,cellar tone,The Conjuring,joseph bishara,7/16/13,58786,20,soundtrack,0.164,0.0236,2,...,0.1170,128.004,4,Horror,0,0,0,1,0,0
2391,Annabelle,The Conjuring,joseph bishara,7/16/13,206346,19,soundtrack,0.138,0.1490,8,...,0.0317,88.096,4,Horror,0,0,0,1,0,0
2392,doll box,The Conjuring,joseph bishara,7/16/13,46600,25,soundtrack,0.409,0.0652,8,...,0.8540,207.542,4,Horror,0,0,0,1,0,0


In [141]:
# Drop features we don't want to use
df = combined_df.drop(columns=['track_name', 'album_name', 'artist_names', 'release_date',\
                      'song_genres','key','movie_genre','popularity'])

df.columns

Index(['duration_ms', 'danceability', 'energy', 'loudness', 'mode',
       'speechiness', 'acousticness', 'instrumentalness', 'liveness',
       'valence', 'tempo', 'time_signature', 'Action', 'Comedy', 'Drama',
       'Horror', 'Romance', 'Sci-Fi'],
      dtype='object')

In [142]:
df.describe()

,duration_ms,danceability,energy,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,Action,Comedy,Drama,Horror,Romance,Sci-Fi
count,2.314000e+03,2314.000000,2314.000000,2314.000000,2314.000000,2314.000000,2314.000000,2314.000000,2314.000000,2314.000000,2314.000000,2314.000000,2314.000000,2314.000000,2314.000000,2314.000000,2314.000000,2314.000000
mean,1.767963e+05,0.322586,0.300851,-17.798736,0.597666,0.055121,0.579665,0.683619,0.149506,0.168306,107.586204,3.733794,0.261884,0.154278,0.136992,0.119274,0.084702,0.242869
std,1.169989e+05,0.203633,0.248699,7.602529,0.490475,0.051594,0.330682,0.340596,0.117117,0.221102,31.771266,0.760577,0.439755,0.361293,0.343913,0.324181,0.278498,0.428910
min,5.546000e+03,0.000000,0.000218,-45.660000,0.000000,0.000000,0.000002,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,1.033300e+05,0.166000,0.089650,-22.861750,0.000000,0.037400,0.278000,0.603000,0.091500,0.036225,81.367750,4.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,1.583105e+05,0.249000,0.231000,-16.994000,1.000000,0.042400,0.663500,0.852000,0.110000,0.055150,104.835500,4.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,2.271755e+05,0.463500,0.461000,-12.245750,1.000000,0.051000,0.883750,0.916000,0.150000,0.202750,128.938750,4.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000
max,3.163192e+06,0.959000,0.984000,-2.591000,1.000000,0.964000,0.995000,0.997000,0.979000,0.970000,216.232000,5.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


## Data Prep / Feature Engineering

In [143]:
# Create dataframes for each binary classification model
df_action = df.drop(columns=['Comedy', 'Drama', 'Horror','Romance','Sci-Fi'])
df_comedy = df.drop(columns=['Action', 'Drama', 'Horror','Romance','Sci-Fi'])
df_drama = df.drop(columns=['Action','Comedy', 'Horror','Romance','Sci-Fi'])
df_horror = df.drop(columns=['Action','Comedy', 'Drama', 'Romance','Sci-Fi'])
df_romance = df.drop(columns=['Action','Comedy', 'Drama', 'Horror','Sci-Fi'])
df_scifi = df.drop(columns=['Action','Comedy', 'Drama', 'Horror','Romance'])

In [144]:
print(df_action.Action.value_counts())
print(df_comedy.Comedy.value_counts())
print(df_horror.Horror.value_counts())
print(df_drama.Drama.value_counts())
print(df_romance.Romance.value_counts())
print(df_scifi['Sci-Fi'].value_counts())

0    1708
1     606
Name: Action, dtype: int64
0    1957
1     357
Name: Comedy, dtype: int64
0    2038
1     276
Name: Horror, dtype: int64
0    1997
1     317
Name: Drama, dtype: int64
0    2118
1     196
Name: Romance, dtype: int64
0    1752
1     562
Name: Sci-Fi, dtype: int64


## Test baseline LGBM on different features

In [145]:
def baseline_GBM_models(df,features_list, y_label):
    auc_final_score = 0
    best_features = []
    best_trans = []
    for vals in features_list:
        genre_movies = df[df[y_label] == 1]
        non_genre_movies = df[df[y_label] == 0]
        
        # Randomly under-sample the majority class (non-genre)
        non_genre_downsampled = resample(non_genre_movies, 
                                          replace=False,     # Sample with replacement
                                          n_samples=len(genre_movies),  # Match the minority class size
                                          random_state=42)   # Set a random seed for reproducibility

        # Concatenate the down-sampled majority class with the original minority class
        df = pd.concat([non_genre_downsampled, genre_movies])
        
        # Define X and y datasets
        X = df.loc[:,vals]
        if 'tempo' in vals:
            X['tempo'] = X['tempo']/240
        y = df[y_label]
        
        # Transform values, if necessary 
        rb_trans = RobustScaler()
        X = rb_trans.fit_transform(X)
        
        # Save scaler to pickle
        lab = y_label.lower()
        with open('{}_genre_rb_scaler.pkl'.format(lab), 'wb') as file:
            pickle.dump(rb_trans, file)

        # Split datasets up
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state = 42)
    
        # SAMPLING
        smote = SMOTE(random_state=42)
        X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)
    
        # Train and run LGBM model
        light_gbm_model = LGBMClassifier(random_state=44,verbose=-1)
        light_gbm_model.fit(X_train, y_train)
        predictions_lgbm = light_gbm_model.predict(X_test)
    
        # Calculate metrics
        #f1_lgbm = f1_score(y_test, predictions_lgbm, average='weighted')
        auc_score = roc_auc_score(y_test, predictions_lgbm, average='weighted')
    
        #print("F1 Score: {:.4f}".format(f1_lgbm))
        #print("ROC AUC Score: {:.4f}".format(auc_score))
        #print("------------")
        
        if auc_score > auc_final_score:
            auc_final_score = auc_score
            best_features = vals
            best_trans = rb_trans
            best_model = light_gbm_model
            best_preds = predictions_lgbm
            
    print("Best AUC Score: {:.4f} with features {}\n".format(auc_final_score,best_features))
    print("\nClassification Report:\n", classification_report(y_test, best_preds))
    
    return [best_model,best_features,best_trans]

In [146]:
# Initial spotify features list
spotify_features_list = ['loudness','tempo','danceability','instrumentalness','acousticness','energy','valence']

# Generate all combinations of features
all_feature_combinations = []
for r in range(1, len(spotify_features_list) + 1):
    feature_combinations = itertools.combinations(spotify_features_list, r)
    all_feature_combinations.extend(list(feature_combinations))
feat_combos = [list(val) for val in all_feature_combinations]

In [147]:
# Test baseline for all movie labels (trying out all feature combinations)
movie_labels = ['Action', 'Comedy', 'Drama', 'Horror','Romance','Sci-Fi']
for label in movie_labels:
    print("Baseline results for {}:".format(label))
    if label == 'Comedy':
        baseline_comedy_model = baseline_GBM_models(df_comedy,feat_combos,label)
    elif label == 'Action':
        baseline_action_model = baseline_GBM_models(df_action,feat_combos,label)
    elif label == 'Drama':
        baseline_drama_model = baseline_GBM_models(df_drama,feat_combos,label)
    elif label == 'Horror':
        baseline_horror_model = baseline_GBM_models(df_horror,feat_combos,label)
    elif label == 'Romance':
        baseline_romance_model = baseline_GBM_models(df_romance,feat_combos,label)
    elif label == 'Sci-Fi':
        baseline_scifi_model = baseline_GBM_models(df_scifi,feat_combos,label)

Baseline results for Action:
Best AUC Score: 0.6356 with features ['tempo', 'instrumentalness', 'acousticness', 'energy', 'valence']


Classification Report:
               precision    recall  f1-score   support

           0       0.67      0.62      0.65       196
           1       0.60      0.65      0.62       168

    accuracy                           0.63       364
   macro avg       0.63      0.64      0.63       364
weighted avg       0.64      0.63      0.64       364

Baseline results for Comedy:
Best AUC Score: 0.7317 with features ['loudness', 'tempo', 'danceability', 'instrumentalness', 'acousticness', 'energy']


Classification Report:
               precision    recall  f1-score   support

           0       0.73      0.82      0.77       117
           1       0.75      0.64      0.69        98

    accuracy                           0.74       215
   macro avg       0.74      0.73      0.73       215
weighted avg       0.74      0.74      0.74       215

Baseline re

In [148]:
# Check feature importances
for label in movie_labels:
    print("Feature Importances for:",label)
    if label == 'Comedy':
        importances = baseline_comedy_model[0].feature_importances_
        gain_importance_df = pd.DataFrame({'Feature': baseline_comedy_model[1], 'Gain': importances})
        print(gain_importance_df.sort_values(by='Gain', ascending=False))
        print("\n")
    elif label == 'Action':
        importances = baseline_action_model[0].feature_importances_
        gain_importance_df = pd.DataFrame({'Feature': baseline_action_model[1], 'Gain': importances})
        print(gain_importance_df.sort_values(by='Gain', ascending=False))
        print("\n")
    elif label == 'Drama':
        importances = baseline_drama_model[0].feature_importances_
        gain_importance_df = pd.DataFrame({'Feature': baseline_drama_model[1], 'Gain': importances})
        print(gain_importance_df.sort_values(by='Gain', ascending=False))
        print("\n")
    elif label == 'Horror':
        importances = baseline_horror_model[0].feature_importances_
        gain_importance_df = pd.DataFrame({'Feature': baseline_horror_model[1], 'Gain': importances})
        print(gain_importance_df.sort_values(by='Gain', ascending=False))
        print("\n")
    elif label == 'Romance':
        importances = baseline_romance_model[0].feature_importances_
        gain_importance_df = pd.DataFrame({'Feature': baseline_romance_model[1], 'Gain': importances})
        print(gain_importance_df.sort_values(by='Gain', ascending=False))
        print("\n")
    elif label == 'Sci-Fi':
        importances = baseline_scifi_model[0].feature_importances_
        gain_importance_df = pd.DataFrame({'Feature': baseline_scifi_model[1], 'Gain': importances})
        print(gain_importance_df.sort_values(by='Gain', ascending=False))
        print("\n")


Feature Importances for: Action
            Feature  Gain
0             tempo   644
1  instrumentalness   611
4           valence   608
3            energy   586
2      acousticness   549


Feature Importances for: Comedy
            Feature  Gain
1             tempo   377
3  instrumentalness   339
2      danceability   337
0          loudness   298
4      acousticness   279
5            energy   266


Feature Importances for: Drama
        Feature  Gain
0      loudness   460
2  acousticness   433
3        energy   408
1  danceability   386


Feature Importances for: Horror
            Feature  Gain
1  instrumentalness   310
3            energy   309
0             tempo   293
4           valence   281
2      acousticness   245


Feature Importances for: Romance
            Feature  Gain
2  instrumentalness   265
3            energy   201
1      danceability   186
0          loudness   160
4           valence   159


Feature Importances for: Sci-Fi
            Feature  Gain
3           

# Build custom models for certain genres

## Drama

In [ ]:
# y_label = 'drama'
# drama_features_list = baseline_drama_model[1]#['tempo', 'instrumentalness', 'acousticness', 'energy', 'valence' ]

# genre_movies = df_drama[df_drama[y_label] == 1]
# non_genre_movies = df_drama[df_drama[y_label] == 0]
        
# # Randomly under-sample the majority class (non-genre)
# non_genre_downsampled = resample(non_genre_movies, 
#                                           replace=False,     # Sample with replacement
#                                           n_samples=len(genre_movies),  # Match the minority class size
#                                           random_state=42)   # Set a random seed for reproducibility

# # Concatenate the down-sampled majority class with the original minority class
# df = pd.concat([non_genre_downsampled, genre_movies])

# # Split into X and y
# X = df_drama.loc[:,drama_features_list]
# if 'tempo' in drama_features_list:
#     X['tempo'] = X['tempo']/240
# y = df_drama[y_label]
        
# # Transform values, if necessary   
# trans_drama = RobustScaler()
# X = trans_drama.fit_transform(X)

    
# # Split datasets up
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state = 42)
    
    
# # Train and run LGBM model
# drama_lgbm_model = LGBMClassifier(random_state=44,verbose=-1)
# drama_lgbm_model.fit(X_train, y_train)
# predictions_lgbm = drama_lgbm_model.predict(X_test)
    
# # Calculate metrics
# f1_lgbm = f1_score(y_test, predictions_lgbm, average='weighted')
# auc_score = roc_auc_score(y_test, predictions_lgbm, average='weighted')
    
# print("F1 Score: {:.4f}".format(f1_lgbm))
# print("ROC AUC Score: {:.4f}".format(auc_score))

# print("\nClassification Report:\n", classification_report(y_test, predictions_lgbm))
# print("\nConfusion Matrix:\n", confusion_matrix(y_test, predictions_lgbm))

## Horror

In [130]:
# y_label = 'horror'
# horror_features_list = baseline_horror_model[1]
# #horror_features_list = ['tempo', 'acousticness', 'valence']

# genre_movies = df_horror[df_horror[y_label] == 1]
# non_genre_movies = df_horror[df_horror[y_label] == 0]
        
# # Randomly under-sample the majority class (non-genre)
# non_genre_downsampled = resample(non_genre_movies, 
#                                           replace=False,     # Sample with replacement
#                                           n_samples=len(genre_movies),  # Match the minority class size
#                                           random_state=42)   # Set a random seed for reproducibility

# # Concatenate the down-sampled majority class with the original minority class
# df = pd.concat([non_genre_downsampled, genre_movies])

# # Split into X and y
# X = df_horror.loc[:,horror_features_list]
# if 'tempo' in horror_features_list:
#     X['tempo'] = X['tempo']/240
# y = df_horror[y_label]
        
# # Transform values, if necessary   
# trans_horror = RobustScaler()
# X = trans_horror.fit_transform(X)

    
# # Split datasets up
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state = 42)
    
    
# # Train and run LGBM model
# horror_lgbm_model = LGBMClassifier(random_state=44,verbose=-1)
# horror_lgbm_model.fit(X_train, y_train)
# predictions_lgbm = horror_lgbm_model.predict(X_test)
    
# # Calculate metrics
# f1_lgbm = f1_score(y_test, predictions_lgbm, average='weighted')
# auc_score = roc_auc_score(y_test, predictions_lgbm, average='weighted')
    
# print("F1 Score: {:.4f}".format(f1_lgbm))
# print("ROC AUC Score: {:.4f}".format(auc_score))

# print("\nClassification Report:\n", classification_report(y_test, predictions_lgbm))
# print("\nConfusion Matrix:\n", confusion_matrix(y_test, predictions_lgbm))

KeyError: 'horror'

## Test on playlists

In [266]:
#playlist
playlist_df = pd.read_csv("../data/raw/on_repeat_sumedh_spotify.csv",",")
playlist_df.head()

/Library/Frameworks/Python.framework/Versions/3.7/lib/python3.7/site-packages/IPython/core/interactiveshell.py:3553: FutureWarning: In a future version of pandas all arguments of read_csv except for the argument 'filepath_or_buffer' will be keyword-only
  exec(code_obj, self.user_global_ns, self.user_ns)


,Spotify ID,Artist IDs,Track Name,Album Name,Artist Name(s),Release Date,Duration (ms),Popularity,Added By,Added At,...,Key,Loudness,Mode,Speechiness,Acousticness,Instrumentalness,Liveness,Valence,Tempo,Time Signature
0,1hR8BSuEqPCCZfv93zzzz9,06HL4z0CvFAxyc27GXpf02,Welcome To New York (Taylor's Version),1989 (Taylor's Version),Taylor Swift,2023-10-26,212600,86,spotify:user:,2023-11-18T09:40:13Z,...,7,-4.830,1,0.0312,0.00961,0.000022,0.3670,0.674,116.980,4
1,3IX0yuEVvDbnqUwMBB3ouC,1McMsnEElThX1knmY4oliG,bad idea right?,GUTS,Olivia Rodrigo,2023-09-08,184783,91,spotify:user:,2023-11-18T09:40:13Z,...,9,-3.446,1,0.0955,0.00193,0.000007,0.0647,0.748,129.979,4
2,6QTGiIuNopQu1iV2aa0fDS,56oDRnqbIiwx4mymNEv7dS,Pink (From Barbie The Album),Pink (From Barbie The Album),Lizzo,2023-07-21,143833,68,spotify:user:,2023-11-18T09:40:13Z,...,1,-7.891,1,0.0427,0.07640,0.000000,0.0763,0.803,114.995,4
3,1vYXt7VSjH9JIM5oRRo7vA,6M2wZ9GZgrQXHCFfjv46we,Dance The Night - From Barbie The Album,Dance The Night (From Barbie The Album),Dua Lipa,2023-05-25,176579,93,spotify:user:,2023-11-18T09:40:13Z,...,11,-4.930,0,0.0480,0.02070,0.000000,0.3290,0.775,110.056,4
4,7G0gBu6nLdhFDPRLc0HdDG,06HL4z0CvFAxyc27GXpf02,Mine (Taylor's Version),Speak Now (Taylor's Version),Taylor Swift,2023-07-07,231706,83,spotify:user:,2023-11-18T09:40:13Z,...,7,-2.846,1,0.0356,0.00444,0.000001,0.1710,0.490,121.080,4


In [267]:
# Update columns names (lowercase, underscore, no special chars)
playlist_df.columns = playlist_df.columns.str.replace(' ','_').str.lower()
playlist_df.columns = playlist_df.columns.str.replace('(','').str.replace(')','')

/Library/Frameworks/Python.framework/Versions/3.7/lib/python3.7/site-packages/ipykernel_launcher.py:3: FutureWarning: The default value of regex will change from True to False in a future version. In addition, single character regular expressions will *not* be treated as literal strings when regex=True.
  This is separate from the ipykernel package so we can avoid doing imports until


In [268]:
# Transform tempo column
playlist_df['tempo'] = playlist_df['tempo']/240

In [269]:
# Get individual features for each model
X_comedy = playlist_df.loc[:,baseline_comedy_model[1]]
X_action = playlist_df.loc[:,baseline_action_model[1]]
X_horror = playlist_df.loc[:,baseline_horror_model[1]]
X_drama = playlist_df.loc[:,baseline_drama_model[1]]
X_romance = playlist_df.loc[:,baseline_romance_model[1]]
X_scifi = playlist_df.loc[:,baseline_scifi_model[1]]

In [270]:
print(X_comedy.columns)
print(X_action.columns)
print(X_horror.columns)
print(X_drama.columns)
print(X_romance.columns)
print(X_scifi.columns)

Index(['loudness', 'tempo', 'danceability', 'instrumentalness', 'acousticness',
       'energy'],
      dtype='object')
Index(['tempo', 'instrumentalness', 'acousticness', 'energy', 'valence'], dtype='object')
Index(['tempo', 'instrumentalness', 'acousticness', 'energy', 'valence'], dtype='object')
Index(['loudness', 'danceability', 'acousticness', 'energy'], dtype='object')
Index(['loudness', 'danceability', 'instrumentalness', 'energy', 'valence'], dtype='object')
Index(['danceability', 'instrumentalness', 'energy', 'valence'], dtype='object')


In [271]:
# Transform datasets
X_comedy = baseline_comedy_model[2].transform(X_comedy)
X_action = baseline_action_model[2].transform(X_action)
X_horror = baseline_horror_model[2].transform(X_horror)
X_drama = baseline_drama_model[2].transform(X_drama)
X_romance = baseline_romance_model[2].transform(X_romance)
X_scifi = baseline_scifi_model[2].transform(X_scifi)

In [272]:
# Predict genre likelihoods and attach back to dataframe
playlist_df['Action'] = baseline_action_model[0].predict_proba(X_action)[:,1]
playlist_df['Comedy'] = baseline_comedy_model[0].predict_proba(X_comedy)[:,1]
playlist_df['Romance'] = baseline_romance_model[0].predict_proba(X_romance)[:,1]
playlist_df['Horror'] = baseline_horror_model[0].predict_proba(X_horror)[:,1]
playlist_df['Drama'] = baseline_drama_model[0].predict_proba(X_drama)[:,1]
playlist_df['Sci-Fi'] = baseline_scifi_model[0].predict_proba(X_scifi)[:,1]

In [273]:
# View condensed results
df_condensed = playlist_df[["track_name","album_name","artist_names","Action","Comedy","Romance","Horror","Drama","Sci-Fi"]]
df_condensed

,track_name,album_name,artist_names,Action,Comedy,Romance,Horror,Drama,Sci-Fi
0,Welcome To New York (Taylor's Version),1989 (Taylor's Version),Taylor Swift,0.177887,0.982999,0.127410,0.029435,0.410165,0.127432
1,bad idea right?,GUTS,Olivia Rodrigo,0.260623,0.572652,0.788350,0.057320,0.197647,0.501376
2,Pink (From Barbie The Album),Pink (From Barbie The Album),Lizzo,0.076558,0.992142,0.227402,0.027430,0.109430,0.070396
3,Dance The Night - From Barbie The Album,Dance The Night (From Barbie The Album),Dua Lipa,0.108931,0.972972,0.836837,0.025599,0.351294,0.087122
4,Mine (Taylor's Version),Speak Now (Taylor's Version),Taylor Swift,0.014604,0.366461,0.745595,0.038490,0.286773,0.341808
5,vampire,GUTS,Olivia Rodrigo,0.073447,0.826714,0.948612,0.339998,0.804836,0.235593
6,Juice,Cuz I Love You,Lizzo,0.125825,0.934113,0.463452,0.020397,0.277227,0.087122
7,favorite kind of high,chemistry,Kelly Clarkson,0.015116,0.970541,0.521082,0.031124,0.317608,0.097925
8,I Know Places (Taylor's Version),1989 (Taylor's Version),Taylor Swift,0.113107,0.784811,0.903332,0.032276,0.246009,0.367501
9,Savage Remix (feat. Beyoncé),Savage Remix (feat. Beyoncé),"Megan Thee Stallion,Beyoncé",0.071001,0.972522,0.323023,0.108065,0.535233,0.156570


In [274]:
# Get means of values
genre_stats = df_condensed.describe().loc['mean',:]
print(genre_stats)

# Get list of median values
median_vals = [df_condensed[val].median() for val in genre_stats.index]
print(median_vals)

Action     0.152495
Comedy     0.877040
Romance    0.488493
Horror     0.110228
Drama      0.484693
Sci-Fi     0.158168
Name: mean, dtype: float64
[0.0952692839539685, 0.9665382441330463, 0.4634516579535305, 0.04790504618478378, 0.3663231081658853, 0.09302245203474803]


In [275]:
# Append to df of means and calculate average of mean/median
stats_df = pd.DataFrame(genre_stats)
stats_df['median'] = median_vals
stats_df = stats_df[['median']].reset_index()
#stats_df

,index,median
0,Action,0.095269
1,Comedy,0.966538
2,Romance,0.463452
3,Horror,0.047905
4,Drama,0.366323
5,Sci-Fi,0.093022


In [276]:
genre_cutoff_dict = {'Action': 0.15, 'Comedy': 0.104, 'Romance': 0.158, 
                               'Horror': 0.156, 'Drama': 0.18, 'Sci-Fi': 0.16}

bad_genres = []
for i,val in stats_df.iterrows():
    if val['median']<genre_cutoff_dict[val['index']]:
        bad_genres.append(val['index'])
print(bad_genres)

['Action', 'Horror', 'Sci-Fi']


## Run model on original training data to get cutoffs

In [228]:
cutoff_df = df.drop(columns=['Action','Comedy','Drama','Horror','Romance','Sci-Fi'])
cutoff_df

,duration_ms,danceability,energy,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature
0,263973,0.194,0.2940,-19.131,1,0.0513,0.605,0.890,0.0862,0.0333,159.468,4
1,123040,0.440,0.1770,-21.577,0,0.0340,0.897,0.925,0.1080,0.0663,124.030,4
2,209586,0.194,0.1530,-20.763,0,0.0471,0.820,0.962,0.1320,0.0338,117.213,4
3,290440,0.224,0.2050,-17.388,0,0.0490,0.566,0.937,0.1070,0.0372,119.046,4
4,107546,0.154,0.6700,-12.670,0,0.0509,0.051,0.664,0.1760,0.0381,84.224,5
...,...,...,...,...,...,...,...,...,...,...,...,...
2389,205880,0.169,0.1380,-25.104,1,0.0436,0.987,0.912,0.0723,0.0265,60.143,4
2390,58786,0.164,0.0236,-38.083,1,0.0438,0.901,0.868,0.1110,0.1170,128.004,4
2391,206346,0.138,0.1490,-21.586,1,0.0443,0.213,0.750,0.0857,0.0317,88.096,4
2392,46600,0.409,0.0652,-22.133,1,0.0939,0.895,0.784,0.2360,0.8540,207.542,4


In [181]:
# Transform tempo column
cutoff_df['tempo'] = cutoff_df['tempo']/240

In [182]:
# Get individual features for each model
X_comedy = cutoff_df.loc[:,baseline_comedy_model[1]]
X_action = cutoff_df.loc[:,baseline_action_model[1]]
X_horror = cutoff_df.loc[:,baseline_horror_model[1]]
X_drama = cutoff_df.loc[:,baseline_drama_model[1]]
X_romance = cutoff_df.loc[:,baseline_romance_model[1]]
X_scifi = cutoff_df.loc[:,baseline_scifi_model[1]]

In [183]:
# Transform datasets
X_comedy = baseline_comedy_model[2].transform(X_comedy)
X_action = baseline_action_model[2].transform(X_action)
X_horror = baseline_horror_model[2].transform(X_horror)
X_drama = baseline_drama_model[2].transform(X_drama)
X_romance = baseline_romance_model[2].transform(X_romance)
X_scifi = baseline_scifi_model[2].transform(X_scifi)

In [184]:
# Predict genre likelihoods and attach back to dataframe
cutoff_df['Action'] = baseline_action_model[0].predict_proba(X_action)[:,1]
cutoff_df['Comedy'] = baseline_comedy_model[0].predict_proba(X_comedy)[:,1]
cutoff_df['Romance'] = baseline_romance_model[0].predict_proba(X_romance)[:,1]
cutoff_df['Horror'] = baseline_horror_model[0].predict_proba(X_horror)[:,1]
cutoff_df['Drama'] = baseline_drama_model[0].predict_proba(X_drama)[:,1]
cutoff_df['Sci-Fi'] = baseline_scifi_model[0].predict_proba(X_scifi)[:,1]

In [194]:
new_cutoff_df = cutoff_df[['Action','Comedy','Romance','Horror','Drama','Sci-Fi']].describe().loc['25%',:]
new_cutoff_df

Action     0.150229
Comedy     0.103528
Romance    0.158170
Horror     0.156116
Drama      0.180486
Sci-Fi     0.159975
Name: 25%, dtype: float64

In [229]:
cutoff_dict = new_cutoff_df.to_dict()
print(cutoff_dict)

{'Action': 0.15022932338300166, 'Comedy': 0.10352849825216309, 'Romance': 0.15816979591228017, 'Horror': 0.1561163451417781, 'Drama': 0.18048647382460353, 'Sci-Fi': 0.1599751312682569}


## Save scaler / model to pickle files

In [230]:
# Save robust scalers to pickle files
with open('action_genre_rb_scaler.pkl', 'wb') as file:
    pickle.dump(baseline_action_model[2], file)

with open('comedy_genre_rb_scaler.pkl', 'wb') as file:
    pickle.dump(baseline_comedy_model[2], file)

with open('drama_genre_rb_scaler.pkl', 'wb') as file:
    pickle.dump(baseline_drama_model[2], file)

with open('romance_genre_rb_scaler.pkl', 'wb') as file:
    pickle.dump(baseline_romance_model[2], file)

with open('horror_genre_rb_scaler.pkl', 'wb') as file:
    pickle.dump(baseline_horror_model[2], file)
    
with open('scifi_genre_rb_scaler.pkl', 'wb') as file:
    pickle.dump(baseline_scifi_model[2], file)

In [231]:
## Save models to pickle files

# save action model
pickle.dump(baseline_action_model[0], open("music_to_action_final.pickle", "wb"))

# save comedy model
pickle.dump(baseline_comedy_model[0], open("music_to_comedy_final.pickle", "wb"))

# save drama model
pickle.dump(baseline_drama_model[0], open("music_to_drama_final.pickle", "wb"))

# save romance model
pickle.dump(baseline_romance_model[0], open("music_to_romance_final.pickle", "wb"))

# save horror model
pickle.dump(baseline_horror_model[0], open("music_to_horror_final.pickle", "wb"))

# save sci-fi model
pickle.dump(baseline_scifi_model[0], open("music_to_scifi_final.pickle", "wb"))